In [1]:
import requests
import json
import re
import uuid
from datetime import datetime
from bs4 import BeautifulSoup
import unicodedata

SITEMAP_URL = "https://www.alternet.org/feeds/sitemaps/sitemap_1.xml"
OUTPUT_PATH = "alternet_200_articles.json"
MAX_ARTICLES = 200
HEADERS = {"User-Agent": "Mozilla/5.0"}

def normalize_text_for_nlp(text):
    if not text:
        return ""

    # Unicode normalization
    text = unicodedata.normalize("NFC", text)

    # Fix common whitespace/codepoint issues
    text = text.replace("\u00A0", " ")
    text = re.sub(r"[\u200B-\u200D\uFEFF]", "", text)

    # Normalize quotes/dashes
    replacements = {
        "\u2018": "'", 
        "\u2019": "'", 
        "\u201C": '"',
        "\u201D": '"',
        "\u2013": "-",
        "\u2014": "-",
        "\u2026": "...",
    }
    for k, v in replacements.items():
        text = text.replace(k, v)

    # Collapse spaces
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()
    
def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def normalize_punctuation(text):
    # Remove space before punctuation
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    # Remove space inside brackets/quotes
    text = re.sub(r"([\(\[\{'\"])\s+", r"\1", text)
    text = re.sub(r"\s+([\)\]\}'\"])", r"\1", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def extract_article_body(soup):
    body_div = soup.find("div", class_="body-description")
    if not body_div:
        return ""

    paragraphs = []

    SKIP_IDS = {
        "pianoFloatingEmailFormContainer",
        "story_page_desktop_incontent_p2_target",
        "story_page_desktop_incontent_p4_target",
        "story_page_desktop_incontent_p10_target",
        "story_page_desktop_incontent_p14_target",
        "alternet_dt_incontent",
    }

    SKIP_CLASSES = {
        "postContent4",
        "postContent10",
        "postContent14",
        "bigcrunch-unit-injected",
        "bigcrunch-unit",
        "shortcode-media",
        "shortcode-media-youtube",
    }

    for p in body_div.find_all("p"):

        # Skip unwanted blocks
        if p.get("class") and any(cls in SKIP_CLASSES for cls in p.get("class")):
            continue
        if p.find_parent(id=SKIP_IDS):
            continue
        if p.find_parent("div", class_=SKIP_CLASSES):
            continue
        if p.find(["iframe", "script", "ins"]):
            continue

        # Skip media blocks
        if p.find("span", class_="rm-shortcode"):
            continue
        if p.find("small", class_=["image-media", "media-caption", "media-photo-credit"]):
            continue

        text = p.get_text(separator=" ", strip=True)

        # Skip small / non-article content
        if len(text.split()) < 6:
            continue

        # Normalize paragraph
        text = normalize_text_for_nlp(text)
        text = normalize_punctuation(text)

        paragraphs.append(text)

    # ---- MATCH TAC FORMAT: paragraph separation ----
    return "\n\n".join(paragraphs)

def get_article_urls():
    response = requests.get(SITEMAP_URL, headers=HEADERS, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "xml")
    return [loc.text for loc in soup.find_all("loc")]

def scrape_article(url: str):
    response = requests.get(url, headers=HEADERS, timeout=10)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Headline from specific structure
    headline = ""
    headline_h1 = soup.find("h1", class_="headline")
    if headline_h1:
        headline = normalize_text_for_nlp(headline_h1.get_text(strip=True))

    # Date published
    date_published = ""
    date_span = soup.find("span", class_="post-date")
    if date_span:
        date_published = date_span.get_text(strip=True)

    # Main body only
    body_text = extract_article_body(soup)
    word_count = len(body_text.split())

    return {
        "article_id": str(uuid.uuid4()),
        "headline": headline,
        "date_published": date_published,
        "body_text": body_text,
        "body_word_count": word_count,
        "outlet": "AlterNet",
        "label": "left",  # Update as needed
        "url": url,
        "processed_at": datetime.utcnow().isoformat(),
    }


def main():
    urls = get_article_urls()
    articles = []

    for url in urls:
        if len(articles) >= MAX_ARTICLES:
            break

        try:
            article = scrape_article(url)

            # Ensure we actually captured article content
            if article["body_word_count"] >= 200:
                articles.append(article)
                print(f"Collected {len(articles)} articles")

        except Exception as e:
            print(f"Skipping {url}: {e}")

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(articles, f, ensure_ascii=False, indent=2)

    print(f"\nSaved {len(articles)} articles to {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

Collected 1 articles
Collected 2 articles
Collected 3 articles
Collected 4 articles
Collected 5 articles
Collected 6 articles
Collected 7 articles
Collected 8 articles
Collected 9 articles
Collected 10 articles
Collected 11 articles
Collected 12 articles
Collected 13 articles
Collected 14 articles
Collected 15 articles
Collected 16 articles
Collected 17 articles
Collected 18 articles
Collected 19 articles
Collected 20 articles
Collected 21 articles
Collected 22 articles
Collected 23 articles
Collected 24 articles
Collected 25 articles
Collected 26 articles
Collected 27 articles
Collected 28 articles
Collected 29 articles
Collected 30 articles
Collected 31 articles
Collected 32 articles
Collected 33 articles
Collected 34 articles
Collected 35 articles
Collected 36 articles
Collected 37 articles
Collected 38 articles
Collected 39 articles
Collected 40 articles
Collected 41 articles
Collected 42 articles
Collected 43 articles
Collected 44 articles
Collected 45 articles
Collected 46 articl